In [0]:
%pip install PyPDF2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#define source directory path
source_path= '/Volumes/agentic_catalog/agentic_schema/customer_service/01_Data_Files/product_docs/'

#list all files in directory
files = dbutils.fs.ls(source_path)

#filter for PDF files only
pdf_files= [f for f in files if f.name.endswith('.pdf')]

print(f"Found{len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  -{pdf.name}")



Found509 PDF files:
  -AccountEase Pro.pdf
  -AccuBooks Pro.pdf
  -AcoustiWave AirBuds Pro.pdf
  -ActiveFit Elite Pro.pdf
  -Advanced Algebra_ Concepts and Applications.pdf
  -AirPure Elite 7000.pdf
  -All-Weather Explorer Jacket.pdf
  -Alpha Z10.pdf
  -AlpinePro Waterproof Jacket.pdf
  -AquaClean Pro 500.pdf
  -Arctic Luxe Thermal Parka.pdf
  -Arctic Shield 360° Winter Jacket.pdf
  -Arctic Shield Unisex Winter Parka.pdf
  -Arctic Thermal Pro Jacket.pdf
  -ArcticGuard Insulated Parka.pdf
  -ArcticShield Deluxe Winter Coat.pdf
  -ArcticTech Insulated Jacket.pdf
  -Aria Modern Bookshelf.pdf
  -ArtisticFlow Pro.pdf
  -Artistry Canvas Masterpiece.pdf
  -Aurora Oak Coffee Table.pdf
  -BBX Cyclone Pro 2023.pdf
  -BeatStream Plus.pdf
  -Biology Today_ Digital Edition.pdf
  -BlazeFlex Comfort Fit Shirt.pdf
  -BlendMaster 4000.pdf
  -BlendMaster Elite 4000.pdf
  -BlendMaster Pro 700.pdf
  -BreezeFlex™ Performance Shirt.pdf
  -BreezeMax AP3000.pdf
  -BrewMaster 3000.pdf
  -BrewMaster 4000.pdf
  

In [0]:
import PyPDF2
import io

def extract_text_from_pdf(file_path):
    """
    Extract text content from a PDF file.

    Args:
        file_path: Full path to the PDF file in DBFS/Volumes

    Returns:
        Extracted text as as string
    """

    try:
        #read pdf file content
        with open(file_path, "rb") as f:
            pdf_bytes = f.read()

        #create a pdf reader object
        pdf_reader= PyPDF2.PdfReader(io.BytesIO(pdf_bytes.encode('latin-1') if isinstance(pdf_bytes,str) else pdf_bytes))

        #extract text from all pages
        text_content = ""
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text_content += page.extract_text() + "\n"

        return text_content.strip()
    
    except Exception as e:
        print(f"Error extracting text from {file_path}: {str(e)}")
        return None
print("Text extraction function defined successfully!")

Text extraction function defined successfully!


In [0]:
#initialize list to store results
product_data = []

#process each pdf file
for pdf_file in pdf_files:
    print(f"Procesing: {pdf_file.name}")

    #extract prod name (remove .pdf extension)
    product_name= pdf_file.name.replace('.pdf','')

    #extract text from pdf
    full_path = f"{source_path}{pdf_file.name}"
    product_doc = extract_text_from_pdf(full_path)

    if product_doc:
        product_data.append({
            'product_name': product_name,
            'product_doc' : product_doc
        })
        print(f"  Successfully extracted {len(product_doc)} characters")
    else:
        print(f"  Failed to extract text")

print(f"\nTotal documents processed: {len(product_data)}")



Procesing: AccountEase Pro.pdf
  Successfully extracted 2674 characters
Procesing: AccuBooks Pro.pdf
  Successfully extracted 3306 characters
Procesing: AcoustiWave AirBuds Pro.pdf
  Successfully extracted 2652 characters
Procesing: ActiveFit Elite Pro.pdf
  Successfully extracted 3194 characters
Procesing: Advanced Algebra_ Concepts and Applications.pdf
  Successfully extracted 2578 characters
Procesing: AirPure Elite 7000.pdf
  Successfully extracted 2964 characters
Procesing: All-Weather Explorer Jacket.pdf
  Successfully extracted 2779 characters
Procesing: Alpha Z10.pdf
  Successfully extracted 3139 characters
Procesing: AlpinePro Waterproof Jacket.pdf
  Successfully extracted 2727 characters
Procesing: AquaClean Pro 500.pdf
  Successfully extracted 3255 characters
Procesing: Arctic Luxe Thermal Parka.pdf
  Successfully extracted 2289 characters
Procesing: Arctic Shield 360° Winter Jacket.pdf
  Successfully extracted 2431 characters
Procesing: Arctic Shield Unisex Winter Parka.pdf

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

#define schema for the dataframe
schema = StructType([
    StructField("product_name", StringType(), False),
    StructField("product_doc", StringType(), True)
])

#create spark dataframe from extraxted data
df = spark.createDataFrame(product_data, schema=schema)

#display the dataframe schema and count
print(f"DataFrame created with {df.count()} rows\n")
df.printSchema()

#write to unity catalog table
table_name = "agentic_catalog.agentic_schema.product_docs"
print(f"\nWriting data to table: {table_name}")

df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f" Successfully created table: {table_name}")


DataFrame created with 509 rows

root
 |-- product_name: string (nullable = false)
 |-- product_doc: string (nullable = true)


Writing data to table: agentic_catalog.agentic_schema.product_docs
 Successfully created table: agentic_catalog.agentic_schema.product_docs


In [0]:
 %sql
 select * from agentic_catalog.agentic_schema.product_docs

product_name,product_doc
AccountEase Pro,"AccountEase Pro Documentation Product Overview AccountEase Pro is a robust online platform designed to simplify personal account management for individuals and businesses alike. With an intuitive interface, users can seamlessly manage their accounts, reset passwords, and maintain high security standards. Getting Started Sign Up Go to the AccountEase Pro website. Click on the 'Sign Up' button. Fill out the registration form with your name, email, and password. Confirm your registration via the verification email. Login Access the portal through the homepage. Enter your registered email and password. Click 'Login'. Navigation The dashboard features user-friendly tabs for quick access to your profile, settings, and support. Account Management Resetting Your Password Click on 'Forgot Password?' on the login page. Enter your email address and submit the form. Check your email for a reset link and follow the instructions. Ensure the link is not expired (valid for 24 hours). Updating Profile Information1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 11. 1. 2. 3. 4. 5. 6. Navigate to 'My Profile'. Edit fields such as your name, email, and phone number. Save changes. Managing Security Settings Go to 'Security Settings' in your account menu. Enable two-factor authentication for enhanced security. Review login history and change your password regularly. Common Troubleshooting Problem: Cannot Reset Password Ensure the reset email hasn't gone to the spam folder. Verify the email you entered is correct and registered. If the link is not working, request a new reset link. Problem: Email Verification Issues Check for a verification email and follow the link inside. Resend the verification email if not received within a few minutes. Problem: Account Locked Accounts may lock after consecutive failed login attempts. Contact support to unlock your account. Advanced Features Custom Integrations Connect AccountEase Pro with third-party applications like Google Drive or Dropbox. Use API keys for seamless data flow. User Customization Personalize the dashboard theme and notification settings.7. 8. 9. 10. 11. 12. 13. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 1. 2. 3. 4. 5. Support and Resources FAQs Visit our FAQ section for quick answers. Customer Support Reach out to support via live chat or email at support@accounteasepro.com. Useful Links Tutorial Videos: www.accounteasepro.com/tutorials User Forum: www.accounteasepro.com/forum For further assistance, refer to our detailed support guides and video tutorials available through the platform. Welcome to AccountEase Pro, where managing your account is made easy and secure.• • • • • • •"
AccuBooks Pro,"AccuBooks Pro: Comprehensive User Documentation Table of Contents Introduction System Requirements Installation Guide Core Features Getting Started Troubleshooting Frequently Asked Questions T echnical Support Updates and Maintenance Compliance and Security 1. Introduction AccuBooks Pro is a cutting-edge accounting software designed for small to medium-sized businesses. With a user-friendly interface and robust functionality, it streamlines financial management tasks such as invoicing, ledger maintenance, tax compliance, payroll, and customized reporting. 2. System Requirements Operating System : Windows 10 or later, macOS 10.14 or later Processor : 1 GHz or faster Memory : 8 GB RAM Storage : 500 MB of available hard disk space Internet : Broadband connection for updates and cloud features 3. Installation Guide Step 1: Download the AccuBooks Pro installer from our official website.1. 2. 3. 4. 5. 6. 7. 8. 9. 10. • • • • • Step 2: Double-click the downloaded file to begin the installation. Step 3: Follow the on-screen instructions to complete the setup. Step 4: Enter your license key when prompted to activate the software. Common Installation Issues: - Error 404 : Re-download the installer as this error indicates a corrupted file. - Error 302 : Ensure your operating system is up-to-date before